In [1]:
import os
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, RocCurveDisplay, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import plotly.express as px
import matplotlib.pyplot as plt
import bioframe as bf
import pysam
import re
from collections import defaultdict

from lib.evaluate_old import load_eval_df, compute_precision_recall_df, plot_precision_recall_curve, parse_vcf, extract_overlap_ids

pd.options.display.max_columns = 100

# Parameters

In [2]:
sample = '906-02'
sample_ = sample.replace('-', '_')
ref = 'hg38'
methods = [('pb', 'svim')]
model_dir = '/confidential/tGenVar/scripts/tGenVar/dicast/models'
model_name = 'RF_100'

# VCF Files

In [3]:
vcfs = {
    'pangini': f'/confidential/tGenVar/maryam/pangenie_run_only_genotyping/{sample}_genotyping.vcf.gz',
    'sniffles': f'/confidential/tGenVar/tech/pb/sv_{ref}/sniffles/{sample_}.vcf.gz',
    'svim': f'/confidential/tGenVar/tech/pb/sv_{ref}/svim/{sample_}/final_results.vcf.gz'
}

# Load Data

In [4]:
# Initial list of confirmed variants
df_labels = df_labels = pd.read_csv(f'/confidential/FamilyR13/DATA/10x/sv_compare/results/{sample}_{ref}/{sample}.decast_input.csv', index_col=0).reset_index(drop=True)

In [5]:
# VCFs other than Illumina
vcf_dfs = []
for tech, method in methods:
    vcf = pysam.VariantFile(vcfs[method])
    vcf_df = parse_vcf(vcf, tech, method, sample)
    vcf_dfs.append(vcf_df)
    
df_vcf = pd.concat(vcf_dfs, ignore_index=True)
df_vcf.loc[df_vcf['type'] == 'DUP:TANDEM', 'type'] = 'DUP'
df_vcf.loc[df_vcf['type'] == 'DUP_INT', 'type'] = 'DUP'

In [6]:
# Dicast eval dataframes
df_eval_del = load_eval_df(model_dir, model_name, 'DEL')
df_eval_ins = load_eval_df(model_dir, model_name, 'INS')
df_eval_inv = load_eval_df(model_dir, model_name, 'INV')
df_eval_dup = load_eval_df(model_dir, model_name, 'DUP')
df_eval = pd.concat([df_eval_del, df_eval_ins, df_eval_inv, df_eval_dup], ignore_index=True)

# Get Confirmed SV IDs

In [7]:
confirmed_ids = defaultdict(list)

for i in range(len(df_labels)):
    sub_graph = df_labels.loc[i, 'sub_graph'][1:-1].split(', ')
    for entry in sub_graph:
        for _, method in methods:
            if entry[1:-1].startswith(method):
                confirmed_ids[method].append(entry[1:-1].split('_')[-1])

In [8]:
df_vcf['confirmed'] = 0 
for _, method in methods:
    df_vcf.loc[df_vcf['id'].isin(confirmed_ids[method]), 'confirmed'] = 1

# Overlap between dicast dataframe and other method dataframes

In [9]:
overlap_ids = []
for svtype in ['DEL', 'INS', 'INV', 'DUP']:
    for tech, method in methods:
        curr_eval_df = df_eval.loc[(df_eval['type'] == svtype), ['id', 'chrom', 'start', 'end', 'size']]
        curr_vcf_df = df_vcf.loc[(df_vcf['type'] == svtype) & (df_vcf['tech'] == tech) & (df_vcf['method'] == method), ['id', 'chrom', 'start', 'end', 'size']]
        curr_vcf_df['end'] = curr_vcf_df['end'].astype(int)
        overlap_ids.append(extract_overlap_ids(curr_eval_df, curr_vcf_df))
overlap_ids = pd.concat(overlap_ids, ignore_index=True)

In [13]:
overlap_ids

,id_1,chrom_1,start_1,end_1,size_1,id_2,chrom_2,start_2,end_2,size_2,distance,diff_start,diff_end,diff_size
0,manta.DEL.6,chr1,1979587,1979880,293.0,svim.DEL.118,chr1,1979610,1979900,290.0,0,23,20,0.989761
1,manta.DEL.292,chr1,228679057,228679394,337.0,svim.DEL.2484,chr1,228679061,228679398,337.0,0,4,4,1.000000
2,manta.DEL.249,chr1,206924435,206924611,176.0,svim.DEL.2326,chr1,206924411,206924597,186.0,0,24,14,0.946237
3,manta.DEL.135,chr1,91448537,91448993,456.0,svim.DEL.1210,chr1,91448541,91448996,455.0,0,4,3,0.997807
4,manta.DEL.279,chr1,224751031,224751093,62.0,svim.DEL.2450,chr1,224751030,224751093,63.0,0,1,0,0.984127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12526,lumpy.DUP.257,chr9,115819293,115828692,9399.0,svim.DUP_TAN.3191,chr9,115819297,115828688,9391.0,0,4,4,0.999149
12527,delly.DUP.5244,chr9,115819294,115828686,9392.0,svim.DUP_TAN.3191,chr9,115819297,115828688,9391.0,0,3,2,0.999894
12528,delly.DUP.5241,chr9,110267689,110275005,7316.0,svim.DUP_TAN.3172,chr9,110267689,110274998,7309.0,0,0,7,0.999043
12529,delly.DUP.9936,chrX,1168999,1169353,354.0,svim.DUP_TAN.6326,chrX,1169002,1169310,308.0,0,3,43,0.870056
